# Part II: Stereo Vision and 3D Reconstruction (KITTI Dataset)

This notebook computes the depth map and 3D point clouds for the KITTI Vision Benchmark Suite.

**Crucial Difference from Kagaru:**
The images provided in the KITTI stereo and odometry datasets are **already rectified**. This means we do not need to compute lens distortion or apply `cv2.stereoRectify()` and `cv2.remap()`. We can directly feed the left and right images into the stereo matching algorithm.

In [1]:
import cv2
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

## 1. Camera Parameters & Q Matrix Initialization
In KITTI, the calibration file (e.g., `calib.txt` or `calib_cam_to_cam.txt`) provides projection matrices `P2` (Left Color Camera) and `P3` (Right Color Camera).

The baseline (B) can be extracted from `P3`: `B = -P3[0, 3] / P3[0, 0]`.
Since images are pre-rectified, we manually construct the Disparity-to-Depth mapping matrix `Q`.

In [10]:
# --- KITTI Calibration Parameters ---
# These are typical values for the KITTI dataset (e.g., sequence 00). 
# You should update these values based on the specific 'calib.txt' of your downloaded sequence.

focal_length = 707.0493  # f_x and f_y are usually the same or very close
cx = 604.0814           # Principal point x
cy = 180.5066           # Principal point y
baseline = 0.5327      # Distance between left and right cameras in meters (approx 54 cm)

# In KITTI, the images are already rectified. Therefore, we don't need to undistort them.
# We can directly construct the Q matrix used for cv2.reprojectImageTo3D
Q = np.array([
    [1.0, 0.0, 0.0, -cx],
    [0.0, 1.0, 0.0, -cy],
    [0.0, 0.0, 0.0, focal_length],
    [0.0, 0.0, 1.0/baseline, 0.0]  # Assuming cx == cx' for rectified KITTI images
], dtype=np.float32)

image_size = (1241, 376) # Typical KITTI image size, adjust if yours is cropped

print("Constructed Q Matrix for KITTI:\n", Q)
print(f"Baseline: {baseline:.4f} meters")

Constructed Q Matrix for KITTI:
 [[   1.           0.           0.        -604.0814   ]
 [   0.           1.           0.        -180.5066   ]
 [   0.           0.           0.         707.0493   ]
 [   0.           0.           1.8772292    0.       ]]
Baseline: 0.5327 meters


## 2. Stereo Matching Configuration
We configure the `StereoSGBM` algorithm. KITTI images are high resolution and have large depth variations (objects very close to the car), so we need a larger `numDisparities` compared to aerial datasets like Kagaru.

In [ ]:
# Configure Semi-Global Block Matching (SGBM)
window_size = 5
min_disp = 0
# KITTI has objects very close to the camera, so disparity can be quite large.
# numDisparities must be divisible by 16. 128 or 160 are good values for KITTI.
num_disp = 16 * 8  

stereo = cv2.StereoSGBM_create(
    minDisparity=min_disp,
    numDisparities=num_disp,
    blockSize=window_size,
    P1=8 * 3 * window_size**2,
    P2=32 * 3 * window_size**2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=32,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)

def write_ply(filename, points, colors):
    """Helper function to save Point Cloud to PLY format"""
    points = points.reshape(-1, 3)
    colors = colors.reshape(-1, 3)
    
    # Filter out points with invalid depth or too far away
    mask = (points[:, 2] > 0) & (points[:, 2] < 100) 
    points = points[mask]
    colors = colors[mask]
    
    if len(points) == 0:
        print(f"Warning: No valid points found for {filename}. Skipping point cloud.")
        return
    
    # Convert BGR to RGB
    colors = cv2.cvtColor(colors.reshape(-1, 1, 3), cv2.COLOR_BGR2RGB).reshape(-1, 3)
    
    header = f'''ply\nformat ascii 1.0\nelement vertex {len(points)}\nproperty float x\nproperty float y\nproperty float z\nproperty uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n'''
    with open(filename, 'w') as f:
        f.write(header)
        for i in range(len(points)):
            f.write(f"{points[i,0]:.4f} {points[i,1]:.4f} {points[i,2]:.4f} {colors[i,0]} {colors[i,1]} {colors[i,2]}\n")


## 3. Processing the Sequence
Looping through the KITTI stereo pairs, calculating disparity, mapping it to a colorful heatmap (JET colormap), and saving the 3D point cloud.

In [12]:
# --- PATHS (UPDATE THESE BASED ON YOUR KITTI DATASET) ---
# KITTI usually provides 'image_2' (left color) and 'image_3' (right color)
left_images_dir = './kitti_samples/1' # <-- UPDATE THIS
right_images_dir = './kitti_samples/2' # <-- UPDATE THIS

left_paths = sorted(glob.glob(os.path.join(left_images_dir, '*.png')))
right_paths = sorted(glob.glob(os.path.join(right_images_dir, '*.png')))

# Limit to 100 frames for testing
num_frames = min(100, len(left_paths))

if num_frames == 0:
    print("WARNING: No images found. Please check your KITTI directory paths.")

# Setup Video Writer (Color visualization for Depth)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# Make sure the image_size matches your actual KITTI images (e.g., 1241x376)
if num_frames > 0:
    sample_img = cv2.imread(left_paths[0])
    h, w = sample_img.shape[:2]
    image_size = (w, h)

out_video_depth = cv2.VideoWriter('kitti_depth_color.mp4', fourcc, 10.0, image_size, True)
out_video_original = cv2.VideoWriter('kitti_original.mp4', fourcc, 10.0, image_size, True)

os.makedirs('kitti_point_clouds', exist_ok=True)
print(f"this is number of image in the left:{len(left_paths)}, right:{len(right_paths)}")
print(f"Processing {num_frames} frames from KITTI...")

for i in range(num_frames):
    print(i)
    # Read Left (Front) and Right images
    img_left_color = cv2.imread(left_paths[i], cv2.IMREAD_COLOR)
    img_right_color = cv2.imread(right_paths[i], cv2.IMREAD_COLOR)
    
    if img_left_color is None or img_right_color is None:
        continue
        
    # Convert to grayscale for SGBM disparity calculation
    img_left_gray = cv2.cvtColor(img_left_color, cv2.COLOR_BGR2GRAY)
    img_right_gray = cv2.cvtColor(img_right_color, cv2.COLOR_BGR2GRAY)
    
    # 1. Compute Disparity (NO RECTIFICATION NEEDED FOR KITTI)
    disparity = stereo.compute(img_left_gray, img_right_gray).astype(np.float32) / 16.0
    
    # 2. Visualize Disparity (JET Colormap)
    # Filter out invalid disparities (negative values) for better visualization
    disp_vis_input = np.maximum(disparity, 0)
    disp_vis = cv2.normalize(disp_vis_input, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    disp_color = cv2.applyColorMap(disp_vis, cv2.COLORMAP_JET)
    
    # Write to videos
    out_video_depth.write(disp_color)
    out_video_original.write(img_left_color)
    
    # 3. Back-project to 3D Point Cloud using constructed Q Matrix
    points_3D = cv2.reprojectImageTo3D(disparity, Q)
    
    # Save the point cloud to PLY format
    write_ply(f"kitti_point_clouds/frame_{i:03d}.ply", points_3D, img_left_color)
    
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{num_frames} frames")

out_video_depth.release()
out_video_original.release()
if num_frames > 0:
    print("Complete! Saved 'kitti_depth_color.mp4', 'kitti_original.mp4', and point clouds.")


this is number of image in the left:62, right:62
Processing 62 frames from KITTI...
0
1
2
3
4
5
6
7
8
9
Processed 10/62 frames
10
11
12
13
14
15
16
17
18
19
Processed 20/62 frames
20
21
22
23
24
25
26
27
28
29
Processed 30/62 frames
30
31
32
33
34
35
36
37
38
39
Processed 40/62 frames
40
41
42
43
44
45
46
47
48
49
Processed 50/62 frames
50
51
52
53
54
55
56
57
58
59
Processed 60/62 frames
60
61
Complete! Saved 'kitti_depth_color.mp4', 'kitti_original.mp4', and point clouds.


In [15]:
import cv2
import os
import numpy as np

output_dir = 'report_samples'
os.makedirs(output_dir, exist_ok=True)

sample_indices = [2, 10, 20] 

print("Generating 3 sample image pairs for the report...")

for idx, frame_idx in enumerate(sample_indices):
    if frame_idx >= len(left_paths):
        print(f"Frame {frame_idx} is out of bounds. Skipping.")
        continue
        
    img_left = cv2.imread(left_paths[frame_idx], cv2.IMREAD_COLOR)
    img_right = cv2.imread(right_paths[frame_idx], cv2.IMREAD_COLOR)
    
    img_left_gray = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
    img_right_gray = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)
    
    disparity = stereo.compute(img_left_gray, img_right_gray).astype(np.float32) / 16.0
    
    disp_vis_input = np.maximum(disparity, 0)
    disp_vis = cv2.normalize(disp_vis_input, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    disp_color = cv2.applyColorMap(disp_vis, cv2.COLORMAP_JET)
    
    orig_name = os.path.join(output_dir, f"orig_sample_{idx+1}.jpg")
    depth_name = os.path.join(output_dir, f"depth_sample_{idx+1}.jpg")
    
    cv2.imwrite(orig_name, img_left)
    cv2.imwrite(depth_name, disp_color)
    
    print(f"Saved Sample {idx+1}: {orig_name} & {depth_name}")

print("All sample images saved successfully in the 'report_samples' folder.")

Generating 3 sample image pairs for the report...
Saved Sample 1: report_samples\orig_sample_1.jpg & report_samples\depth_sample_1.jpg
Saved Sample 2: report_samples\orig_sample_2.jpg & report_samples\depth_sample_2.jpg
Saved Sample 3: report_samples\orig_sample_3.jpg & report_samples\depth_sample_3.jpg
All sample images saved successfully in the 'report_samples' folder.
